# Assignment 2: The Dirty Ledger

**Course:** ICA-F26, The Prices Desk
**Input file:** `teaching_prices_messy.csv`
**Method:** The raw data is loaded and left unchanged. A working copy is created, quality checks are run, and every decision is recorded. No record is deleted and no value is imputed.

## Planned decisions

| Potential issue | Decision | Reason |
|---|---|---|
| Missing values | Investigate; do not fill | Filling would invent prices or categories |
| Exact duplicate rows | Check before removing | Similar rows are not necessarily duplicates |
| Repeated date, item and city combinations | Flag | One observation per combination is expected |
| Unparseable dates | Flag; do not replace | A guessed date changes the observation |
| Non-numeric or non-positive prices | Investigate and flag | A price must be numeric and positive |
| Inconsistent units or labels | Standardize only if the meaning is identical | Different units must not be combined |
| Unusual prices | Flag; do not delete | An unusual value may be genuine |
| Source label | Preserve | It describes what the observations represent |

## Verdicts on common cleaning code

| Proposed action | Verdict | Reason |
|---|---|---|
| Read the CSV with pandas | KEEP | Required to inspect the data |
| Make a separate working copy | KEEP | Protects the raw input |
| Convert `date` to datetime, invalid values coerced to missing | MODIFY | Failures are counted and reviewed, not silently accepted |
| Convert `price` to numeric | MODIFY | Conversion failures are counted before any decision |
| Drop every row containing a missing value | REJECT | Blanket deletion can remove valid observations |
| Drop all duplicate rows | MODIFY | Exact and key duplicates are checked first |
| Remove rows with unusually high or low prices | REJECT | No verified error rule supports deleting an observation |
| Replace missing prices with the mean | REJECT | Mean imputation invents values and distorts variation |
| Standardize units and labels | MODIFY | Applied only where the meaning is identical |
| Overwrite the original CSV | REJECT | The source must remain available for audit |
| Report row counts before and after | KEEP | Makes any record loss visible |

## Verification

The code cell below reports structure, types, missing values, duplicates, price and date validity, category values, the number of rows per item against the expected number, and a before-and-after comparison.

In [4]:
import pandas as pd
source_name = "teaching_prices_messy.csv"
raw = pd.read_csv(source_name)
df = raw.copy(deep=True)
print("Source:", source_name)
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())
display(raw.dtypes.to_frame("dtype"))
display(raw.describe(include="all").T)
for col in ["item", "city", "unit", "data_label"]:
    if col in raw.columns:
        print(col, raw[col].dropna().unique().tolist())
price_num = pd.to_numeric(df["price"], errors="coerce")
date_parsed = pd.to_datetime(df["date"], errors="coerce")
checks = {
    "rows": len(df),
    "columns": len(df.columns),
    "missing_cells": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "duplicate_date_item_city": int(df.duplicated(subset=["date", "item", "city"], keep=False).sum()),
    "non_numeric_prices": int(price_num.isna().sum() - df["price"].isna().sum()),
    "non_positive_prices": int((price_num <= 0).sum()),
    "unparseable_dates": int(date_parsed.isna().sum() - df["date"].isna().sum()),
}
display(pd.Series(checks, name="result").to_frame())
print("Missing values per column:")
display(raw.isna().sum().to_frame("missing"))
df["price"] = price_num
df["date"] = date_parsed
for col in ["unit", "data_label", "item", "city"]:
    if col in df.columns:
        display(df[col].value_counts(dropna=False).to_frame("count"))
expected = df["city"].nunique() * df["date"].nunique()
print("Expected rows per item:", expected)
display(df["item"].value_counts().to_frame("actual_rows"))
summary = pd.DataFrame({
    "before": [len(raw), int(raw.isna().sum().sum()), int(raw.duplicated().sum())],
    "after": [len(df), int(df.isna().sum().sum()), int(df.duplicated().sum())],
}, index=["rows", "missing_cells", "exact_duplicate_rows"])
summary["change"] = summary["after"] - summary["before"]
display(summary)
display(df["price"].describe().to_frame("price_after"))

Source: teaching_prices_messy.csv
Shape: (97, 6)
Columns: ['date', 'item', 'city', 'price', 'unit', 'data_label']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


,dtype
date,object
item,object
city,object
price,object
unit,object
data_label,object


,count,unique,top,freq
date,97,12,2026-01-08,9
item,97,5,Staple A,24
city,97,2,City 1,49
price,96,70,202.5,5
unit,97,1,PKR per kg (illustrative),97
data_label,97,1,ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,97


item ['Staple A', 'Oil B', 'Pulse C', 'Sweetener D', 'oil b']
city ['City 1', 'City 2']
unit ['PKR per kg (illustrative)']
data_label ['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']


,result
rows,97
columns,6
missing_cells,1
exact_duplicate_rows,1
duplicate_date_item_city,2
non_numeric_prices,1
non_positive_prices,0
unparseable_dates,0


... (6 outputs left out)

Expected rows per item: 24


,actual_rows
item,
Staple A,24
Oil B,24
Pulse C,24
Sweetener D,24
oil b,1


,before,after,change
rows,97,97,0
missing_cells,1,2,1
exact_duplicate_rows,1,1,0


,price_after
count,95.000000
mean,305.621053
std,1008.261424
min,99.500000
25%,135.500000
50%,200.000000
75%,301.000000
max,9999.000000
